# EDQ `rules_assignments_m` Configuration & Parameter Update

This notebook is designed for the current EDQ metadata model.

### What it does
1. Reads `rules_assignments_m`, `rules_m`, `assets_m`, and `asset_columns_m`.
2. Normalizes `target_type` from the actual assignment relationships:
   - current asset + no target column → `ASSET`
   - current asset + target column → `COLUMN`
   - target asset + no target asset column → `CROSS_ASSET`
   - target asset + target asset column → `CROSS_ASSET_COLUMN`
3. Detects stale `target_column_id` values.
4. Reads `rules_m.parameter_definition` and builds a required-parameter worklist.
5. Automatically fills only values that can be derived safely from metadata.
6. Leaves business-dependent parameters unresolved instead of guessing.
7. Shows a preview.
8. Applies changes only when `APPLY_CHANGES = True`.

**Important:** keep `APPLY_CHANGES = False` for the first run.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json
from functools import reduce

CATALOG = "mastercard_card_241614"
SCHEMA = "default"

ASSIGNMENTS = f"{CATALOG}.{SCHEMA}.rules_assignments_m"
RULES = f"{CATALOG}.{SCHEMA}.rules_m"
ASSETS = f"{CATALOG}.{SCHEMA}.assets_m"
COLUMNS = f"{CATALOG}.{SCHEMA}.asset_columns_m"

# SAFETY SWITCH
APPLY_CHANGES = False

# Known, user-validated parameter configuration from the current T03 test.
# Keep this only if assignment 35 still represents the same configuration.
MANUAL_PARAMETER_OVERRIDES = {
    35: {
        "source_timestamp_column": "INITIATED_AT",
        "ingestion_timestamp_column": "SETTLED_AT",
        "time_unit": "MINUTE"
    }
}

print("APPLY_CHANGES =", APPLY_CHANGES)


APPLY_CHANGES = False


In [0]:
assignments_df = spark.table(ASSIGNMENTS)
rules_df = spark.table(RULES)
assets_df = spark.table(ASSETS)
columns_df = spark.table(COLUMNS)

print("Assignments :", assignments_df.count())
print("Rules       :", rules_df.count())
print("Assets      :", assets_df.count())
print("Columns     :", columns_df.count())


Assignments : 742
Rules       : 73
Assets      : 28
Columns     : 255


In [0]:
# ------------------------------------------------------------
# 1. Normalize target_type from actual assignment relationships
# ------------------------------------------------------------

normalized_df = (
    assignments_df
    .withColumn(
        "derived_target_type",
        F.when(
            F.col("target_asset_id").isNotNull() &
            F.col("target_asset_column_id").isNotNull(),
            F.lit("CROSS_ASSET_COLUMN")
        )
        .when(
            F.col("target_asset_id").isNotNull(),
            F.lit("CROSS_ASSET")
        )
        .when(
            F.col("target_column_id").isNotNull(),
            F.lit("COLUMN")
        )
        .otherwise(F.lit("ASSET"))
    )
)

display(
    normalized_df.groupBy(
        "target_type",
        "derived_target_type"
    ).count().orderBy(
        "target_type",
        "derived_target_type"
    )
)


target_type,derived_target_type,count
COLUMN,COLUMN,94
COLUMN / CROSS-COLUMN,COLUMN,4
COLUMN / TABLE,COLUMN,196
TABLE,ASSET,280
TABLE / COLUMN,ASSET,168


In [0]:
# ------------------------------------------------------------
# 2. Detect stale target_column_id values
# ------------------------------------------------------------

valid_column_ids = columns_df.select("column_id").distinct()

stale_columns_df = (
    assignments_df.alias("a")
    .filter(F.col("a.target_column_id").isNotNull())
    .join(
        valid_column_ids.alias("c"),
        F.col("a.target_column_id") == F.col("c.column_id"),
        "left_anti"
    )
    .select(
        "assignment_id",
        "asset_id",
        "rule_code",
        "target_column_id",
        "target_type"
    )
)

print("Stale target_column_id rows:", stale_columns_df.count())
display(stale_columns_df.orderBy("asset_id", "assignment_id"))


Stale target_column_id rows: 293


assignment_id,asset_id,rule_code,target_column_id,target_type
1,1,A01,1,COLUMN / TABLE
2,1,A02,1,COLUMN
3,1,B01,1,COLUMN / CROSS-COLUMN
4,1,B02,1,COLUMN / CROSS-COLUMN
5,1,B08,1,COLUMN / CROSS-COLUMN
6,1,C01,2,COLUMN / TABLE
7,1,C02,2,COLUMN / TABLE
8,1,C03,2,COLUMN
9,1,C04,2,COLUMN / TABLE
11,1,C06,2,COLUMN / TABLE


In [0]:
# ------------------------------------------------------------
# 3. Read parameter definitions and build a parameter worklist
# ------------------------------------------------------------

rule_defs = (
    rules_df
    .select(
        "rule_code",
        "rule_type",
        "parameter_definition"
    )
    .collect()
)

parameter_specs = {}

for row in rule_defs:
    definition = row["parameter_definition"]

    if not definition:
        parameter_specs[row["rule_code"]] = []
        continue

    try:
        parsed = json.loads(definition)
        parameter_specs[row["rule_code"]] = parsed.get("parameters", [])
    except Exception as e:
        parameter_specs[row["rule_code"]] = [{
            "name": "__INVALID_PARAMETER_DEFINITION__",
            "control": "invalid",
            "required": True,
            "error": str(e)
        }]

print("Rules with parameter definitions:",
      sum(bool(v) for v in parameter_specs.values()))


Rules with parameter definitions: 47


In [0]:
# ------------------------------------------------------------
# 4. Safe automatic parameter derivation
#
# We only derive values when the metadata itself provides the answer.
# We do NOT guess business rules such as:
#   - required column sets
#   - reference/master assets
#   - SLA conditions
#   - baseline row counts
#   - allowed values
#   - regexes
#   - source/ingestion timestamp pairs
# ------------------------------------------------------------

def get_required_specs(rule_code):
    return [
        p for p in parameter_specs.get(rule_code, [])
        if p.get("required", False)
    ]


# Pull PK information from metadata JSON where available.
# The current design stores source PK information inside metadata.
pk_candidates_df = (
    columns_df
    .withColumn(
        "is_primary_key_meta",
        F.get_json_object(
            F.col("metadata"),
            "$.is_primary_key"
        )
    )
    .withColumn(
        "is_primary_key_meta_bool",
        F.when(
            F.lower(F.col("is_primary_key_meta")) == "true",
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .filter(F.col("is_primary_key_meta_bool") == True)
    .select(
        "asset_id",
        "column_id",
        "column_name",
        "ordinal_position"
    )
)

display(
    pk_candidates_df
    .orderBy("asset_id", "ordinal_position")
)


asset_id,column_id,column_name,ordinal_position


In [0]:
# ------------------------------------------------------------
# 5. Build proposed parameter_values
# ------------------------------------------------------------

# Start with existing parameter_values.
# Manual overrides take precedence.
#
# Safe automatic rules:
#   I04 -> primary_key_column only when exactly one PK is defined.
#   U04 -> primary_key_column only when exactly one PK is defined.
#   U05 -> composite_key_columns when one or more PK columns are
#          explicitly marked in metadata.
#
# Everything else remains unresolved unless supplied in
# MANUAL_PARAMETER_OVERRIDES.

pk_rows = (
    pk_candidates_df
    .orderBy("asset_id", "ordinal_position")
    .collect()
)

pk_by_asset = {}
for r in pk_rows:
    pk_by_asset.setdefault(r["asset_id"], []).append(r["column_name"])


def safe_json(value):
    return json.dumps(value, separators=(",", ":"))


def proposed_parameters(row):
    assignment_id = row["assignment_id"]
    asset_id = row["asset_id"]
    rule_code = row["rule_code"]

    # Explicit user configuration wins.
    if assignment_id in MANUAL_PARAMETER_OVERRIDES:
        return safe_json(MANUAL_PARAMETER_OVERRIDES[assignment_id])

    existing = row["parameter_values"]

    # Do not overwrite already-populated configuration.
    if existing is not None and str(existing).strip() != "":
        return existing

    pks = pk_by_asset.get(asset_id, [])

    if rule_code in ("I04", "U04") and len(pks) == 1:
        return safe_json({
            "primary_key_column": pks[0]
        })

    if rule_code == "U05" and len(pks) >= 1:
        return safe_json({
            "composite_key_columns": pks
        })

    return None


proposed_schema = T.StructType([
    T.StructField("assignment_id", T.LongType(), False),
    T.StructField("proposed_parameter_values", T.StringType(), True)
])

proposed_rows = []

for r in assignments_df.select(
    "assignment_id",
    "asset_id",
    "rule_code",
    "parameter_values"
).collect():

    proposed_rows.append((
        int(r["assignment_id"]),
        proposed_parameters(r)
    ))

proposed_df = spark.createDataFrame(
    proposed_rows,
    proposed_schema
)

preview_df = (
    normalized_df.alias("a")
    .join(
        proposed_df.alias("p"),
        "assignment_id",
        "left"
    )
    .select(
        "assignment_id",
        "asset_id",
        "rule_code",
        "target_column_id",
        "target_asset_id",
        "target_asset_column_id",
        "target_type",
        "derived_target_type",
        "parameter_values",
        "proposed_parameter_values"
    )
)

display(
    preview_df
    .filter(
        (F.col("target_type") != F.col("derived_target_type")) |
        (
            F.coalesce(
                F.col("parameter_values"),
                F.lit("")
            )
            !=
            F.coalesce(
                F.col("proposed_parameter_values"),
                F.lit("")
            )
        )
    )
    .orderBy("asset_id", "assignment_id")
)


assignment_id,asset_id,rule_code,target_column_id,target_asset_id,target_asset_column_id,target_type,derived_target_type,parameter_values,proposed_parameter_values
1,1,A01,1,null,null,COLUMN / TABLE,COLUMN,null,null
3,1,B01,1,null,null,COLUMN / CROSS-COLUMN,COLUMN,null,null
4,1,B02,1,null,null,COLUMN / CROSS-COLUMN,COLUMN,null,null
5,1,B08,1,null,null,COLUMN / CROSS-COLUMN,COLUMN,null,null
6,1,C01,2,null,null,COLUMN / TABLE,COLUMN,null,null
7,1,C02,2,null,null,COLUMN / TABLE,COLUMN,null,null
9,1,C04,2,null,null,COLUMN / TABLE,COLUMN,null,null
11,1,C06,2,null,null,COLUMN / TABLE,COLUMN,null,null
14,1,K02,1,null,null,COLUMN / CROSS-COLUMN,COLUMN,null,null
39,1,U01,1,null,null,COLUMN / TABLE,COLUMN,null,null


In [0]:
# ------------------------------------------------------------
# 6. Required-parameter worklist
# ------------------------------------------------------------

work_rows = []

assignment_rows = (
    assignments_df
    .select(
        "assignment_id",
        "asset_id",
        "rule_code",
        "target_column_id",
        "target_asset_id",
        "target_asset_column_id"
    )
    .collect()
)

for r in assignment_rows:

    specs = get_required_specs(r["rule_code"])

    if not specs:
        continue

    for spec in specs:

        name = spec.get("name")
        control = spec.get("control")

        reason = "BUSINESS_VALUE_REQUIRED"

        if control == "asset_select" and r["target_asset_id"] is None:
            reason = "TARGET_ASSET_REQUIRED"

        elif control in (
            "column_select",
            "column_multi_select",
            "ordered_column_select"
        ) and r["target_column_id"] is None:
            reason = "COLUMN_SELECTION_REQUIRED"

        elif control == "schema_definition":
            reason = "EXPECTED_SCHEMA_CONFIGURATION"

        elif control == "number":
            reason = "NUMERIC_CONFIGURATION_REQUIRED"

        elif control in ("expression", "expression_multi"):
            reason = "SQL_EXPRESSION_REQUIRED"

        elif control == "datetime":
            reason = "DATETIME_CONFIGURATION_REQUIRED"

        elif control == "multi_value":
            reason = "VALUE_SET_REQUIRED"

        elif control == "regex":
            reason = "REGEX_REQUIRED"

        elif control == "select":
            reason = "SELECT_OPTION_REQUIRED"

        work_rows.append((
            int(r["assignment_id"]),
            int(r["asset_id"]),
            r["rule_code"],
            name,
            control,
            reason
        ))

work_schema = T.StructType([
    T.StructField("assignment_id", T.LongType(), False),
    T.StructField("asset_id", T.LongType(), False),
    T.StructField("rule_code", T.StringType(), False),
    T.StructField("parameter_name", T.StringType(), False),
    T.StructField("control", T.StringType(), True),
    T.StructField("status", T.StringType(), False)
])

worklist_df = spark.createDataFrame(work_rows, work_schema)

display(
    worklist_df.orderBy(
        "asset_id",
        "assignment_id",
        "parameter_name"
    )
)


assignment_id,asset_id,rule_code,parameter_name,control,status
1,1,A01,reference_asset,asset_select,TARGET_ASSET_REQUIRED
1,1,A01,reference_column,column_select,BUSINESS_VALUE_REQUIRED
1,1,A01,source_column,column_select,BUSINESS_VALUE_REQUIRED
2,1,A02,lookup_condition,expression,SQL_EXPRESSION_REQUIRED
2,1,A02,reference_asset,asset_select,TARGET_ASSET_REQUIRED
2,1,A02,reference_column,column_select,BUSINESS_VALUE_REQUIRED
3,1,B01,condition,expression,SQL_EXPRESSION_REQUIRED
3,1,B01,required_condition,expression,SQL_EXPRESSION_REQUIRED
4,1,B02,condition,expression,SQL_EXPRESSION_REQUIRED
4,1,B02,validation_condition,expression,SQL_EXPRESSION_REQUIRED


In [0]:
# ------------------------------------------------------------
# 7. Final configuration validation
# ------------------------------------------------------------

validation_df = (
    preview_df
    .withColumn(
        "target_type_status",
        F.when(
            F.col("target_type") == F.col("derived_target_type"),
            F.lit("OK")
        ).otherwise(F.lit("CHANGE_REQUIRED"))
    )
    .withColumn(
        "parameter_status",
        F.when(
            F.col("proposed_parameter_values").isNotNull() &
            (F.trim(F.col("proposed_parameter_values")) != ""),
            F.lit("CONFIGURED")
        ).otherwise(
            F.lit("MISSING")
        )
    )
)

display(
    validation_df.groupBy(
        "target_type_status",
        "parameter_status"
    ).count()
)


target_type_status,parameter_status,count
OK,MISSING,94
CHANGE_REQUIRED,CONFIGURED,1
CHANGE_REQUIRED,MISSING,647


In [0]:
# ------------------------------------------------------------
# 8. APPLY CHANGES
# ------------------------------------------------------------

if not APPLY_CHANGES:
    print("DRY RUN ONLY")
    print("No rows have been modified.")
    print("Set APPLY_CHANGES = True only after reviewing the previews.")
else:

    proposed_df.createOrReplaceTempView(
        "dq_assignment_parameter_updates"
    )

    spark.sql(f'''
        MERGE INTO {ASSIGNMENTS} AS target
        USING (
            SELECT
                assignment_id,
                proposed_parameter_values
            FROM dq_assignment_parameter_updates
        ) AS source
        ON target.assignment_id = source.assignment_id

        WHEN MATCHED THEN UPDATE SET
            target.target_type =
                CASE
                    WHEN target.target_asset_id IS NOT NULL
                         AND target.target_asset_column_id IS NOT NULL
                        THEN 'CROSS_ASSET_COLUMN'

                    WHEN target.target_asset_id IS NOT NULL
                        THEN 'CROSS_ASSET'

                    WHEN target.target_column_id IS NOT NULL
                        THEN 'COLUMN'

                    ELSE 'ASSET'
                END,

            target.parameter_values =
                CASE
                    WHEN source.proposed_parameter_values IS NOT NULL
                        THEN source.proposed_parameter_values
                    ELSE target.parameter_values
                END,

            target.updated_at = current_timestamp(),

            target.change_reason =
                CASE
                    WHEN source.proposed_parameter_values IS NOT NULL
                        THEN 'Metadata-driven target type / parameter configuration update'
                    ELSE target.change_reason
                END
    ''')

    print("Changes applied successfully.")


DRY RUN ONLY
No rows have been modified.
Set APPLY_CHANGES = True only after reviewing the previews.


In [0]:
# ------------------------------------------------------------
# 9. POST-UPDATE CHECK
# ------------------------------------------------------------

display(
    spark.table(ASSIGNMENTS)
    .groupBy("target_type")
    .count()
    .orderBy("target_type")
)

display(
    spark.table(ASSIGNMENTS)
    .filter(
        F.col("parameter_values").isNotNull()
    )
    .select(
        "assignment_id",
        "asset_id",
        "rule_code",
        "target_type",
        "parameter_values"
    )
    .orderBy("asset_id", "assignment_id")
)


target_type,count
COLUMN,94
COLUMN / CROSS-COLUMN,4
COLUMN / TABLE,196
TABLE,280
TABLE / COLUMN,168


assignment_id,asset_id,rule_code,target_type,parameter_values
